# Experiment 3: Decision Tree - ID3 Algorithm
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Demonstrate the working of the decision-tree based **ID3** algorithm on an appropriate data set (PlayTennis), build the tree and use it to classify a new sample.

## Theory
* **Entropy:** `H(S) = - sum p_i log2 p_i`
* **Information Gain:** `Gain(S, A) = H(S) - sum (|S_v|/|S|) H(S_v)`
* ID3 greedily picks the attribute with the **highest information gain** at each node and recurses on each branch until the node is pure or no attributes remain.

In [1]:
import math
import pandas as pd
from collections import Counter

rows = [
 ['Sunny','Hot','High','Weak','No'],        ['Sunny','Hot','High','Strong','No'],
 ['Overcast','Hot','High','Weak','Yes'],    ['Rain','Mild','High','Weak','Yes'],
 ['Rain','Cool','Normal','Weak','Yes'],     ['Rain','Cool','Normal','Strong','No'],
 ['Overcast','Cool','Normal','Strong','Yes'],['Sunny','Mild','High','Weak','No'],
 ['Sunny','Cool','Normal','Weak','Yes'],    ['Rain','Mild','Normal','Weak','Yes'],
 ['Sunny','Mild','Normal','Strong','Yes'],  ['Overcast','Mild','High','Strong','Yes'],
 ['Overcast','Hot','Normal','Weak','Yes'],  ['Rain','Mild','High','Strong','No'],
]
df = pd.DataFrame(rows, columns=['Outlook','Temperature','Humidity','Wind','PlayTennis'])
df

,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [2]:
def entropy(series):
    counts = Counter(series)
    total = len(series)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

def info_gain(data, attr, target):
    total_entropy = entropy(data[target])
    weighted = 0.0
    for value, subset in data.groupby(attr):
        weighted += len(subset) / len(data) * entropy(subset[target])
    return total_entropy - weighted

target = 'PlayTennis'
attributes = [c for c in df.columns if c != target]
print(f'Entropy of the whole data set: {entropy(df[target]):.4f}\n')
for a in attributes:
    print(f'Information gain of {a:12s}: {info_gain(df, a, target):.4f}')

Entropy of the whole data set: 0.9403

Information gain of Outlook     : 0.2467
Information gain of Temperature : 0.0292
Information gain of Humidity    : 0.1518
Information gain of Wind        : 0.0481


In [3]:
def id3(data, target, attrs):
    labels = data[target]
    if labels.nunique() == 1:                 # pure node
        return labels.iloc[0]
    if not attrs:                             # no attributes left
        return labels.mode()[0]
    gains = {a: info_gain(data, a, target) for a in attrs}
    best = max(gains, key=gains.get)
    tree = {best: {}}
    for value in data[best].unique():
        subset = data[data[best] == value]
        tree[best][value] = id3(subset, target, [a for a in attrs if a != best])
    return tree

tree = id3(df, target, attributes)

def print_tree(t, indent=''):
    if not isinstance(t, dict):
        print(indent + '-> ' + str(t))
        return
    attr = next(iter(t))
    for value, sub in t[attr].items():
        print(f'{indent}[{attr} = {value}]')
        print_tree(sub, indent + '    ')

print('Decision tree built by ID3:\n')
print_tree(tree)

Decision tree built by ID3:

[Outlook = Sunny]
    [Humidity = High]
        -> No
    [Humidity = Normal]
        -> Yes
[Outlook = Overcast]
    -> Yes
[Outlook = Rain]
    [Wind = Weak]
        -> Yes
    [Wind = Strong]
        -> No


In [4]:
def classify(t, sample):
    while isinstance(t, dict):
        attr = next(iter(t))
        t = t[attr][sample[attr]]
    return t

new_samples = [
    {'Outlook': 'Sunny',    'Temperature': 'Cool', 'Humidity': 'High',   'Wind': 'Strong'},
    {'Outlook': 'Overcast', 'Temperature': 'Mild', 'Humidity': 'High',   'Wind': 'Weak'},
    {'Outlook': 'Rain',     'Temperature': 'Mild', 'Humidity': 'Normal', 'Wind': 'Strong'},
]
for s in new_samples:
    print(s, '->', classify(tree, s))

# training accuracy of the learned tree
pred = df.apply(lambda r: classify(tree, r), axis=1)
print('\nTraining accuracy:', (pred == df[target]).mean())

{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'} -> No
{'Outlook': 'Overcast', 'Temperature': 'Mild', 'Humidity': 'High', 'Wind': 'Weak'} -> Yes
{'Outlook': 'Rain', 'Temperature': 'Mild', 'Humidity': 'Normal', 'Wind': 'Strong'} -> No

Training accuracy: 1.0


## Result
`Outlook` has the highest information gain, so it becomes the root. The tree classifies all 14 training examples correctly and was used to classify new, unseen samples.